# MLflow Tracing Demo on OpenShift AI

An interactive chat app demonstrating **end-to-end GenAI observability** with MLflow Tracing. Every conversation turn, LLM call, and tool invocation is automatically captured as a trace, grouped into sessions for review and feedback.

## What it does

- **Chat** with an LLM served on OpenShift AI (vLLM / KServe) via an OpenAI-compatible endpoint.
- **GitHub tool calling** — ask about a repo and the assistant fetches live info through a **GitHub MCP server**.
- **Automatic tracing** — each turn is logged to MLflow (with nested LLM + tool spans), grouped by session and user.
- **History & feedback** — reload past conversations from traces, and submit ratings/comments back to MLflow.

## How to use

1. Run the cells in order (restart the kernel after the first `pip install`).
2. Type in the chat box and press **Enter** / **Send**. Ask about a GitHub repo (e.g. *"Tell me about vllm-project/vllm"*) to trigger the tool.
3. Use **＋ New Conversation**, **Refresh**, and **Load & Continue** to manage sessions, and the **feedback** panel to rate replies.
4. View all traces in the **MLflow UI**.

## How to configure

Edit the **`Config` class** in the configuration cell — the single place for all settings (`MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT`, `LLM_BASE_URL` / `LLM_API_KEY` / `LLM_MODEL`, `GITHUB_MCP_URL` / `GITHUB_PAT`, `USER_ID`). Values can also be set via environment variables.

> 📌 **Important:** If the chat UI doesn't render or the widgets appear unresponsive, **restart the kernel, refresh the webpage, and run the cells again**. The first `pip install` sets up `ipywidgets`, which only becomes fully functional after a fresh kernel and page reload.

# Install dependencies

In [1]:
!unzip -q wheels.zip -d wheels

replace wheels/wheels/aiohappyeyeballs-2.7.1-py3-none-any.whl? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [3]:
import os
import sys

print(f"Python version in Workbench: {sys.version.split()[0]}")
print(f"Current working directory:   {os.getcwd()}")

# Search for all .whl files in the current directory and subdirectories
found_wheels = []
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".whl"):
            found_wheels.append(os.path.join(root, file))

print(f"\nTotal .whl files found: {len(found_wheels)}")
if found_wheels:
    print("Sample wheels found:")
    for w in found_wheels[:8]:
        print(" ", w)
    if len(found_wheels) > 8:
        print(f"  ... and {len(found_wheels) - 8} more.")
else:
    print("⚠️ No .whl files found anywhere in the current directory tree!")
# added for checking local .whl files

Python version in Workbench: 3.12.9
Current working directory:   /opt/app-root/src

Total .whl files found: 113
Sample wheels found:
  ./wheels/wheels/aiohappyeyeballs-2.7.1-py3-none-any.whl
  ./wheels/wheels/aiohttp-3.14.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
  ./wheels/wheels/aiosignal-1.4.0-py3-none-any.whl
  ./wheels/wheels/alembic-1.20.1-py3-none-any.whl
  ./wheels/wheels/annotated_doc-0.0.5-py3-none-any.whl
  ./wheels/wheels/annotated_types-0.8.0-py3-none-any.whl
  ./wheels/wheels/anyio-4.15.1-py3-none-any.whl
  ./wheels/wheels/asttokens-3.0.2-py3-none-any.whl
  ... and 105 more.


In [ ]:
%pip install --no-index --find-links=wheels/wheels --quiet "mlflow>=3.11" openai ipywidgets markdown
# Restart the kernel after the first install so ipywidgets renders correctly.
# edited version for using local .whl files

In [48]:
for pkg in ["mlflow", "openai", "ipywidgets", "markdown"]:
    try:
        mod = __import__(pkg)
        print(f"✓ {pkg}: {getattr(mod, '__version__', 'installed')}")
    except ImportError:
        print(f"✗ {pkg}: NOT installed")

✓ mlflow: 3.17.2
✓ openai: 3.36.1
✓ ipywidgets: 8.1.9
✓ markdown: 3.11.2


Trace(trace_id='tr-25b6a9a66a36079732e65e73c8c2f106')

In [49]:
import mlflow
import openai
import ipywidgets
import markdown

print("MLflow:", mlflow.__version__)
print("OpenAI:", openai.__version__)
print("Environment ready!")
# confirm wheel files are ready

MLflow: 3.17.2
OpenAI: 3.36.1
Environment ready!


# Configuration (change credentials, URLs and identifiers here ONLY)

In [50]:
import os


class Config:
    # --- MLflow (OpenShift AI 3.4 managed MLflow) --------------------------
    # Find the route with:  oc get route -n <mlflow-namespace>
    MLFLOW_TRACKING_URI = os.getenv(
        "MLFLOW_TRACKING_URI",
        "mlflow-redhat-ods-applications.apps.ocp4.telleruat",
    )
    MLFLOW_EXPERIMENT = os.getenv(
        "MLFLOW_EXPERIMENT", 
        "tracing-demo"
    )
    MLFLOW_INSECURE_TLS = True   # demo/dev only
    # If MLflow uses the cluster's internal service CA, you may instead set:
    #   REQUESTS_CA_BUNDLE = "/path/to/ca.crt"
    REQUESTS_CA_BUNDLE = None

    # --- LLM endpoint (model served in OpenShift AI, e.g. vLLM / KServe) ---
    LLM_BASE_URL = os.getenv(
        "LLM_BASE_URL",
        "https://maas.apps.ocp4.telleruat/v1",
    )
    LLM_API_KEY = os.getenv(
        "LLM_API_KEY",
        "sk-oai-1IAOwZp8H1p7meNUW_10K2RNod9riPFdFd2X4hsPJCzW5YhRz2XEyx7BA3l9A",
    )
    LLM_MODEL = os.getenv(
        "LLM_MODEL", 
        "publishers/llm-serving/models/deepseek-v4"
    )

    # --- GitHub MCP server -------------------------------------------------
     # GITHUB_MCP_URL = os.getenv(
     #    "GITHUB_MCP_URL", 
     #    "https://api.githubcopilot.com/mcp/"
     # )
     # GITHUB_PAT = os.getenv(
     #    "GITHUB_PAT", 
     #    "your-github-personal-access-token"
     # )

    # --- Identity / UI -----------------------------------------------------
    USER_ID = os.getenv(
        "DEMO_USER", 
        "Red Hat"
    )
    OPENSHIFT_LOGO_URL = (
        "https://ulx.hu/wp-content/uploads/2023/01/OpenShift-logo-transparent.png"
    )

    # --- MLflow trace metadata keys (standard) -----------------------------
    SESSION_META_KEY = "mlflow.trace.session"
    USER_META_KEY = "mlflow.trace.user"

In [51]:
# --- Extra step to obtain kubernetes token to avoid using kubernetes library ----
import os
import mlflow
from mlflow import MlflowClient

# 1. Remove the flag that triggers the missing 'kubernetes' Python library
os.environ.pop("MLFLOW_TRACKING_AUTH", None)

# 2. Read the pod's service account token mounted by OpenShift
token_file = "/var/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(token_file):
    with open(token_file, "r") as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

# 3. Read the pod namespace (used for workspace routing)
ns_file = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(ns_file):
    with open(ns_file, "r") as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

# 4. If your cluster route uses self-signed certificates, bypass TLS verification
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

# Now execute your original lines:
mlflow.set_tracking_uri(Config.MLFLOW_TRACKING_URI)
mlflow.set_experiment(Config.MLFLOW_EXPERIMENT)
mlflow.openai.autolog()

mlflow_client = MlflowClient()
print("MLflow client connected successfully!")

MLflow client connected successfully!


In [52]:
# Imports & environment setup

In [53]:
import json
import uuid
import html
import asyncio
import threading
import datetime as dt

import urllib3
import mlflow
from mlflow import MlflowClient
from openai import OpenAI
# from mcp import ClientSession
# from mcp.client.streamable_http import streamablehttp_client

import ipywidgets as widgets
from IPython.display import display

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

if Config.MLFLOW_INSECURE_TLS:
    os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
if Config.REQUESTS_CA_BUNDLE:
    os.environ["REQUESTS_CA_BUNDLE"] = Config.REQUESTS_CA_BUNDLE

print("This cell runs with no error!")
    

This cell runs with no error!


# Markdown rendering helper

In [54]:
"""Markdown -> HTML renderer with graceful fallbacks."""

def _build_markdown_renderer():
    try:
        import markdown as _md_lib
        exts = ["fenced_code", "tables", "sane_lists", "nl2br"]
        return lambda text: _md_lib.markdown(str(text), extensions=exts)
    except Exception:
        pass

    try:
        from markdown_it import MarkdownIt
        _mdit = MarkdownIt("commonmark", {"breaks": True, "html": False}).enable("table")
        return lambda text: _mdit.render(str(text))
    except Exception:
        pass

    # Last resort: escape + preserve newlines.
    return lambda text: html.escape(str(text)).replace("\n", "<br>")


md_to_html = _build_markdown_renderer()


# MLflow & LLM clients

In [55]:
mlflow.set_tracking_uri(Config.MLFLOW_TRACKING_URI)
mlflow.set_experiment(Config.MLFLOW_EXPERIMENT)
mlflow.openai.autolog()   # auto-trace every OpenAI-compatible LLM call

mlflow_client = MlflowClient()

_experiment = mlflow.get_experiment_by_name(Config.MLFLOW_EXPERIMENT)
EXPERIMENT_ID = _experiment.experiment_id if _experiment else "0"

llm = OpenAI(base_url=Config.LLM_BASE_URL, api_key=Config.LLM_API_KEY)

print("MLflow tracking URI:", Config.MLFLOW_TRACKING_URI)
print("Experiment         :", Config.MLFLOW_EXPERIMENT)


MLflow tracking URI: https://rh-ai.apps.ocp4.telleruat/mlflow
Experiment         : tracing-demo


# GitHub MCP tool

In [56]:
"""GitHub MCP integration + the tool exposed to the LLM."""

def _run_async(coro):
    """Run a coroutine on a fresh event loop.

    Safe from both the kernel thread and background worker threads
    (a worker thread has no event loop bound to it by default).
    """
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(coro)
    finally:
        loop.close()
        asyncio.set_event_loop(None)


async def _mcp_call(tool_name: str, arguments: dict) -> str:
    headers = {"Authorization": f"Bearer {Config.GITHUB_PAT}"}
    async with streamablehttp_client(Config.GITHUB_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            return "\n".join(getattr(b, "text", str(b)) for b in result.content)


@mlflow.trace(span_type="TOOL", name="github_mcp.get_repo_info")
def get_github_repo_info(owner: str, repo: str) -> str:
    """Tool exposed to the LLM. Calls the GitHub MCP server for repo info.

    The @mlflow.trace decorator records this as a child TOOL span
    (inputs + outputs) under the chat trace.
    """
    mlflow.update_current_trace(tags={"github.owner": owner, "github.repo": repo})
    try:
        return _run_async(_mcp_call("search_repositories", {"query": f"repo:{owner}/{repo}"}))
    except Exception as e:
        return f"GitHub MCP call failed: {e}"


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_github_repo_info",
            "description": "Fetch information about a GitHub repository (stars, "
                           "description, language, etc.) via the GitHub MCP server. "
                           "Use this whenever the user asks about a GitHub repo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner/org"},
                    "repo":  {"type": "string", "description": "Repository name"},
                },
                "required": ["owner", "repo"],
            },
        },
    }
]

# Maps tool name -> python callable, so chat_turn stays generic.
TOOL_DISPATCH = {"get_github_repo_info": get_github_repo_info}


# Chat turn (one trace per turn)

In [57]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. If the user asks about a GitHub repository, "
    "call the get_github_repo_info tool. Otherwise answer normally."
)

@mlflow.trace(name="chat_turn", span_type="AGENT")
def chat_turn(history: list, user_message: str, session_id: str, user_id: str) -> str:
    """One conversational turn = one trace, stamped with an immutable session_id
    so MLflow groups every turn of the conversation under a single Session."""
    mlflow.update_current_trace(metadata={
        Config.SESSION_META_KEY: session_id,
        Config.USER_META_KEY: user_id,
    })

    messages = [{"role": "system", "content": ""}]
    messages += history
    messages.append({"role": "user", "content": user_message})

    # resp = llm.chat.completions.create(
    #     model=Config.LLM_MODEL, messages=messages, tools=TOOLS, tool_choice="auto"
    # )
    resp = llm.chat.completions.create(
        model=Config.LLM_MODEL,
        messages=[
            {"role": "user", "content": user_message }
        ],
        temperature=0.7,
        # DO NOT pass tools=tools or tool_choice here as Github mcp is skipped
    )
    msg = resp.choices[0].message

    if msg.tool_calls:
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            tool_fn = TOOL_DISPATCH.get(tc.function.name)
            if tool_fn:
                args = json.loads(tc.function.arguments or "{}")
                tool_result = tool_fn(**args)        # traced TOOL span
            else:
                tool_result = f"Unknown tool: {tc.function.name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result,
            })
        resp = llm.chat.completions.create(model=Config.LLM_MODEL, messages=messages)
        msg = resp.choices[0].message

    return msg.content or ""

print("This cell runs with no error!")


This cell runs with no error!


# Trace / session history helpers

In [58]:
"""Read past conversations back out of MLflow traces."""

_PAGE_SIZE = 100   # tracking server rejects large max_results


def _search_traces_page(filter_string, page_token, page_size):
    """One page of Trace objects. Prefers the new `locations` kwarg,
    falls back to the deprecated `experiment_ids` on older MLflow."""
    kwargs = dict(filter_string=filter_string, max_results=page_size, page_token=page_token)
    try:
        return mlflow_client.search_traces(locations=[EXPERIMENT_ID], **kwargs)
    except TypeError:
        return mlflow_client.search_traces(experiment_ids=[EXPERIMENT_ID], **kwargs)


def search_traces(filter_string=None, max_results=1000):
    """Collect Trace objects across pages, up to max_results."""
    results, token = [], None
    while len(results) < max_results:
        page_size = min(_PAGE_SIZE, max_results - len(results))
        try:
            page = _search_traces_page(filter_string, token, page_size)
        except Exception as e:
            if not results:
                raise
            print(f"[history] stopped paging early: {e}")
            break
        results.extend(list(page))
        token = getattr(page, "token", None)
        if not token or len(page) == 0:
            break
    return results


# --- Trace field extractors ---------------------------------------------------

def _trace_metadata(trace) -> dict:
    info = getattr(trace, "info", None)
    if info is None:
        return {}
    for attr in ("trace_metadata", "request_metadata"):
        d = getattr(info, attr, None)
        if d:
            return d
    return getattr(info, "tags", {}) or {}


def _trace_ts(trace) -> int:
    info = getattr(trace, "info", None)
    if info is None:
        return 0
    for attr in ("timestamp_ms", "request_time", "start_time", "start_time_ms"):
        v = getattr(info, attr, None)
        if v is None:
            continue
        if isinstance(v, dt.datetime):
            return int(v.timestamp() * 1000)
        try:
            return int(v)
        except (TypeError, ValueError):
            continue
    return 0


def _trace_request_response(trace):
    data = getattr(trace, "data", None)
    req_raw = getattr(data, "request", None) if data else None
    resp_raw = getattr(data, "response", None) if data else None
    try:
        req = json.loads(req_raw) if isinstance(req_raw, str) else (req_raw or {})
    except Exception:
        req = {}
    return req, resp_raw


def _trace_user_message(trace) -> str:
    req, _ = _trace_request_response(trace)
    return str(req.get("user_message", ""))[:48] if isinstance(req, dict) else ""


# --- Public history API --------------------------------------------------------

def list_previous_sessions(max_sessions: int = 50):
    """Return [(session_id, last_ts_ms, first_user_message)] for the menu."""
    grouped = {}   # sid -> {"first_ts", "last_ts", "label"}
    for trace in search_traces(max_results=1000):
        sid = _trace_metadata(trace).get(Config.SESSION_META_KEY)
        if not sid:
            continue
        ts = _trace_ts(trace)
        g = grouped.get(sid)
        if g is None:
            grouped[sid] = {"first_ts": ts, "last_ts": ts,
                            "label": _trace_user_message(trace) or "(conversation)"}
        else:
            g["last_ts"] = max(g["last_ts"], ts)
            if ts < g["first_ts"]:                  # keep earliest message as label
                g["first_ts"] = ts
                g["label"] = _trace_user_message(trace) or g["label"]

    ordered = sorted(grouped.items(), key=lambda kv: kv[1]["last_ts"], reverse=True)
    return [(sid, g["last_ts"], g["label"]) for sid, g in ordered[:max_sessions]]


def load_session_history(session_id: str) -> list:
    """Rebuild a past conversation's messages from its traces (oldest first)."""
    safe_sid = session_id.replace("'", "\\'")
    traces = search_traces(
        filter_string=f"metadata.`{Config.SESSION_META_KEY}` = '{safe_sid}'",
        max_results=1000,
    )
    # Fallback: if metadata filtering isn't supported server-side, filter locally.
    if not traces:
        traces = [t for t in search_traces(max_results=1000)
                  if _trace_metadata(t).get(Config.SESSION_META_KEY) == session_id]

    history = []
    for trace in sorted(traces, key=_trace_ts):      # oldest -> newest
        req, resp_raw = _trace_request_response(trace)
        user_msg = req.get("user_message", "") if isinstance(req, dict) else ""
        try:
            answer = json.loads(resp_raw) if isinstance(resp_raw, str) else resp_raw
            answer = "" if answer is None else (answer if isinstance(answer, str) else str(answer))
        except Exception:
            answer = str(resp_raw or "")
        if user_msg:
            history.append({"role": "user", "content": user_msg})
        if answer:
            history.append({"role": "assistant", "content": answer})
    return history


# Chat UI rendering

In [44]:
"""WhatsApp-style chat rendering."""

# Conversation state shared across callbacks.
state = {"session_id": str(uuid.uuid4()), "history": []}

# Scoped styling so Markdown elements look right inside the bubbles.
_MD_CSS = """
<style>
.md-body p { margin: 4px 0; }
.md-body > :first-child { margin-top:0; }
.md-body > :last-child { margin-bottom:0; }
.md-body pre {
    background:#272822; color:#f8f8f2; padding:10px; border-radius:6px;
    overflow:auto; font-size:0.85em;
}
.md-body code { background:rgba(0,0,0,0.08); padding:1px 4px; border-radius:4px; font-size:0.9em; }
.md-body pre code { background:transparent; padding:0; }
.md-body table { border-collapse:collapse; margin:6px 0; }
.md-body th, .md-body td { border:1px solid #ccc; padding:4px 8px; }
.md-body th { background:#f0f0f0; }
.md-body ul, .md-body ol { margin:4px 0 4px 22px; }
.md-body blockquote { border-left:3px solid #ccc; margin:4px 0; padding:2px 10px; color:#555; }
</style>
"""

# WhatsApp-style wallpaper: a subtle dotted pattern over the classic beige.
_WALLPAPER = (
    "background-color:#e5ddd5;"
    "background-image:radial-gradient(rgba(0,0,0,0.05) 1px,transparent 1px);"
    "background-size:18px 18px;"
)


def _fmt_user(text: str) -> str:
    # User input is raw text: escape it and keep line breaks.
    return html.escape(str(text)).replace("\n", "<br>")


def _fmt_assistant(text: str) -> str:
    # Assistant replies are rendered as Markdown.
    return md_to_html(text)


def _render_bubble(message: dict) -> str:
    is_user = message["role"] == "user"
    if is_user:
        body, bubble_bg, align, name, name_color = (
            _fmt_user(message["content"]), "#dcf8c6", "flex-end", "🧑 You", "#1f7a4d")
    else:
        body, bubble_bg, align, name, name_color = (
            _fmt_assistant(message["content"]), "#ffffff", "flex-start", "🤖 Assistant", "#0b62d6")

    return (
        f"<div style='display:flex;justify-content:{align};margin:6px 0'>"
        f"<div class='md-body' style='max-width:78%;background:{bubble_bg};"
        f"padding:8px 12px;border-radius:12px;box-shadow:0 1px 1px rgba(0,0,0,0.15);"
        f"font-size:0.95em;color:#111;word-wrap:break-word;overflow-wrap:anywhere'>"
        f"<div style='font-size:1.3em;font-weight:700;color:{name_color};"
        f"margin-bottom:3px'>{name}</div>"
        f"{body}"
        f"</div></div>"
    )


def _render():
    session_label.value = (
        f"<b>Session:</b> <code>{state['session_id']}</code> &nbsp;"
        f"<span style='color:#888'>({len(state['history']) // 2} turns)</span>"
    )
    rows = "".join(_render_bubble(m) for m in state["history"])
    chat_log.value = (
        _MD_CSS
        + f"<div id='chat-scroll' style='height:300px;overflow:auto;"
        f"border:1px solid #ccc;padding:14px;border-radius:8px;{_WALLPAPER}"
        f"margin-bottom:10px'>"
        + rows
        + "</div>"
    )


# Widgets & event handlers

In [45]:
"""Chat controls + callbacks."""

# --- Header banner ------------------------------------------------------------
header = widgets.HTML(
    value=(
        "<div style='display:flex;align-items:center;gap:18px;"
        "padding:18px 22px;margin-bottom:14px;border-radius:12px;"
        "background:linear-gradient(90deg,#1a1a1a 0%,#2b2b2b 100%);"
        "border:1px solid #000'>"
        f"<img src='{Config.OPENSHIFT_LOGO_URL}' alt='OpenShift logo' "
        "style='height:56px;width:auto;flex:0 0 auto'/>"
        "<div style='display:flex;flex-direction:column;line-height:1.15'>"
        "<span style='font-size:1.9em;font-weight:700;color:#ffffff;"
        "font-family:\"Red Hat Display\",Arial,sans-serif'>"
        "MLflow tracing demo on OpenShift AI</span>"
        "<span style='font-size:0.95em;color:#bbb;margin-top:2px'>"
        "Interactive chat &amp; conversation history</span>"
        "</div></div>"
    )
)

# --- Chat widgets -------------------------------------------------------------
session_label = widgets.HTML()
chat_log = widgets.HTML()
text_in = widgets.Text(
    placeholder="Ask anything, or about a GitHub repo…",
    continuous_update=False,                  # fires on Enter / blur
    layout=widgets.Layout(width="70%"),
)
send_btn = widgets.Button(description="Send", button_style="primary")
new_btn = widgets.Button(description="＋ New Conversation", button_style="success")
hist_dd = widgets.Dropdown(
    options=[("— previous conversations —", None)],
    layout=widgets.Layout(width="48%"),
)
load_btn = widgets.Button(description="Load & Continue")
refresh_btn = widgets.Button(description="Refresh", layout=widgets.Layout(width="100px"))


# --- Callbacks ----------------------------------------------------------------
def _refresh_history(_=None):
    opts = [("— previous conversations —", None)]
    for sid, ts, preview in list_previous_sessions():
        try:
            when = dt.datetime.fromtimestamp(ts / 1000).strftime("%m-%d %H:%M")
        except Exception:
            when = ""
        opts.append((f"{when}  |  {preview}", sid))
    hist_dd.options = opts


def _on_send(_=None):
    msg = text_in.value.strip()
    if not msg:
        return
    text_in.value = ""
    send_btn.disabled = text_in.disabled = True
    state["history"].append({"role": "user", "content": msg})
    state["history"].append({"role": "assistant", "content": "…thinking"})
    _render()

    def worker():
        try:
            answer = chat_turn(
                history=state["history"][:-2],     # everything before this turn
                user_message=msg,
                session_id=state["session_id"],
                user_id=Config.USER_ID,
            )
        except Exception as e:
            answer = f"⚠️ Error: {e}"
        state["history"][-1] = {"role": "assistant", "content": answer}
        send_btn.disabled = text_in.disabled = False
        _render()

    threading.Thread(target=worker, daemon=True).start()


def _on_new(_=None):
    state["session_id"] = str(uuid.uuid4())
    state["history"] = []
    _render()


def _on_load(_=None):
    sid = hist_dd.value
    if not sid:
        return
    state["session_id"] = sid
    state["history"] = load_session_history(sid)
    _render()


send_btn.on_click(_on_send)
text_in.observe(
    lambda ch: _on_send() if ch["name"] == "value" and ch["new"] else None,
    names="value",
)
new_btn.on_click(_on_new)
load_btn.on_click(_on_load)
refresh_btn.on_click(_refresh_history)


# Feedback panel

In [46]:
"""User feedback -> MLflow."""

fb_text = widgets.Textarea(
    value="",
    placeholder="Tell us what worked well or what could be improved…",
    layout=widgets.Layout(width="99%", height="80px"),
)
fb_rating = widgets.Dropdown(
    options=[("👍 Positive", "positive"),
             ("😐 Neutral", "neutral"),
             ("👎 Negative", "negative")],
    value="positive",
    description="Rating:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto", margin="3"),
)
fb_btn = widgets.Button(
    description="Submit feedback",
    button_style="success",
    icon="paper-plane",
    tooltip="Send your feedback to MLflow for this session",
)
fb_status = widgets.HTML(value="")


def _submit_feedback(_btn):
    comment = fb_text.value.strip()
    if not comment:
        fb_status.value = "<span style='color:#c0392b'>⚠️ Please enter some feedback text first.</span>"
        return

    session_id = state["session_id"]
    rating = fb_rating.value
    try:
        # Prefer attaching feedback to the session's most recent trace.
        trace_id = mlflow.get_last_active_trace_id()

        if trace_id is not None and hasattr(mlflow, "log_feedback"):
            # MLflow 3.x GenAI feedback API — attaches feedback directly to a trace.
            mlflow.log_feedback(
                trace_id=trace_id,
                name="user_feedback",
                value=rating,
                rationale=comment,
                source=mlflow.entities.AssessmentSource(
                    source_type="HUMAN", source_id=Config.USER_ID
                ),
            )
            where = f"trace <code>{trace_id}</code>"
        else:
            # Fallback: record feedback as its own short MLflow run, keyed by session.
            with mlflow.start_run(run_name=f"feedback-{session_id}"):
                mlflow.set_tags({
                    "type": "user_feedback",
                    "session_id": session_id,
                    "rating": rating,
                })
                mlflow.log_text(comment, "feedback.txt")
                mlflow.log_param("rating", rating)
            where = f"session <code>{session_id}</code>"

        fb_status.value = f"<span style='color:#1f7a4d'>✅ Thanks! Feedback recorded for {where}.</span>"
        fb_text.value = ""
    except Exception as e:
        fb_status.value = f"<span style='color:#c0392b'>❌ Could not save feedback: {e}</span>"


fb_btn.on_click(_submit_feedback)

feedback_panel = widgets.VBox(
    [
        widgets.HTML("<b style='font-size:1.05em'>💬 Send feedback</b>"),
        fb_text,
        widgets.HBox([fb_rating, fb_btn]),
        fb_status,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd", border_radius="8px",
        padding="12px", margin="10px 0 0 0",
    ),
)


# Launch the app

> 📌 **Important:** If the chat UI doesn't render or the widgets appear unresponsive, **restart the kernel, refresh the webpage, and run the cells again**. The first `pip install` sets up `ipywidgets`, which only becomes fully functional after a fresh kernel and page reload.

In [47]:
_refresh_history()
_render()

display(widgets.VBox([
    header,
    widgets.HBox([new_btn, hist_dd, refresh_btn, load_btn]),
    session_label,
    chat_log,
    widgets.HBox([text_in, send_btn]),
    feedback_panel,
]))
